# Rapport d'évaluation RAGAS — NBA Analyst AI

Ce notebook retrace les évaluations RAGAS successives du pipeline RAG (`ragas_part/evaluate_ragas.py`, dataset de 15 questions, seuil de succès `SCORE_THRESHOLD = 0.7`) ainsi que les améliorations apportées au système entre chaque itération.

## Évolution des scores RAGAS (moyennes globales, 15 questions)

| Étape | faithfulness | context precision | context recall | answer relevancy |
|---|---|---|---|---|
| Prototype initial | 0.392 | 0.122 | 0.060 | 0.365 |
| Après sécurisation schémas Pydantic | 0.594 | 0.089 | 0.163 | 0.624 |
| Meilleur résultat (version actuelle) | 0.500 | 0.350 | 0.361 | 0.948 |

In [ ]:
import pandas as pd

scores = pd.DataFrame(
    [
        {"etape": "Prototype initial", "faithfulness": 0.392, "context_precision": 0.122, "context_recall": 0.060, "answer_relevancy": 0.365},
        {"etape": "Après sécurisation SQL + Pydantic", "faithfulness": 0.594, "context_precision": 0.089, "context_recall": 0.163, "answer_relevancy": 0.624},
        {"etape": "Meilleur résultat (version actuelle)", "faithfulness": 0.5004, "context_precision": 0.35, "context_recall": 0.3611, "answer_relevancy": 0.9484},
    ]
).set_index("etape")
scores

## Améliorations apportées au système RAG

### 1. Environnement et reproductibilité 
- Environnement reproductible géré avec `uv` (`pyproject.toml` + `uv.lock` figés).
- Configuration centralisée dans `utils/config.py`, secrets dans `.env` (non versionné).
- Mise en place du pipeline d'évaluation RAGAS : dataset de questions/réponses attendues (`ragas_dataset.json`), génération des réponses et contextes via le pipeline réel, calcul des métriques.

### 2. Fiabilisation du pipeline avec Pydantic 
- Schémas Pydantic à chaque étape du pipeline (`utils/schemas.py`) : `RawDocument` → `CleanedDocument` → `Chunk` → `EmbeddedChunk` → `SearchResult`, remplaçant des `dict` non typés.
- Validation des entrées utilisateur (`RAGQuery` : longueur, non-vide) avant envoi au LLM.
- Sortie du LLM forcée dans un schéma structuré (`RAGAnswer`) via Pydantic AI, au lieu de texte libre à parser.
- Documents invalides capturés et ignorés (`ValidationError`) sans interrompre le reste de l'indexation.
- Ajout de l'observabilité Pydantic Logfire sur les appels de l'agent.

### 3. Ajustement des hyperparamètres du LLM 
- Les hyperparamètres du LLM n'étaient jusque-là pas explicitement contrôlés : l'agent tournait avec les valeurs par défaut de l'API Mistral (température élevée), ce qui produisait des réponses trop créatives/peu fiables pour un cas d'usage factuel (statistiques NBA).
- Ajout de `LLM_TEMPERATURE`, `LLM_TOP_P`, `LLM_MAX_TOKENS` dans `utils/config.py`, transmis explicitement à l'agent via `model_settings` (`utils/chatbot.py`).
- Température abaissée à `0.2` (quasi déterministe) pour privilégier la fidélité aux contextes récupérés plutôt que la créativité, `top_p = 0.8` et `max_tokens = 800`.

### 4. Sécurisation de l'outil NL→SQL pour les statistiques NBA 
- Chargement des statistiques NBA dans PostgreSQL (`Sql_db/load_excel_to_db.py`).
- Génération de requêtes SQL en langage naturel via LangChain (prompt few-shot + schéma PostgreSQL introspecté).
- Sécurisation de l'outil : requêtes restreintes aux `SELECT` uniquement (`is_safe_select`), limite de lignes forcée (`enforce_row_limit`).
- Erreurs SQL (requête invalide, colonne inexistante, base injoignable) capturées et renvoyées comme message lisible plutôt que de lever une exception.
- Ajout des noms de colonnes dans la réponse SQL pour un contexte plus exploitable par le LLM juge RAGAS.

### 5. Observabilité et logging 
- Centralisation de la configuration du logging dans `utils/observability.py`.
- Logfire configuré de façon conditionnelle (local par défaut, cloud si `LOGFIRE_TOKEN` défini).
- Pont entre le logger standard Python et Logfire (tous les `logging.info/warning/error` remontent automatiquement).
- Runs RAGAS journalisés directement dans Logfire (`ragas_evaluation_run`) au lieu d'un fichier JSON local.

### 6. Organisation, tests et outillage
- Réorganisation du projet en modules dédiés (`ragas_part/`, `Sql_db/`) pour ne pas surcharger `utils/`.
- Makefile pour lancer facilement les scripts (`make index`, `make ragas-refresh`, `make test`, etc.).
- Tests unitaires (`tests/unit`), fonctionnels (`tests/functional`) et d'intégration (`tests/integration`, skippés sans identifiants réels).
- Contexte de l'outil SQL intégré à la génération du dataset RAGAS, pour évaluer aussi les réponses basées sur les statistiques structurées.

## Constats

- `context_precision` et `context_recall` restent les points faibles : les contextes récupérés ne couvrent pas encore correctement les besoins des questions statistiques complexes (agrégations, comparaisons entre équipes/joueurs).
- `answer_relevancy` a fortement progressé (0.365 → 0.948) grâce à la sécurisation de l'outil SQL, aux schémas Pydantic et à l'enrichissement du contexte (noms de colonnes, outil SQL dans le dataset RAGAS).
- Aucun run n'a encore atteint le seuil global (`SCORE_THRESHOLD = 0.7`) sur les quatre métriques simultanément : la prochaine piste d'amélioration porte sur la récupération de contexte (`context_precision`/`context_recall`).

## Recommandations

Sur la base des constats ci-dessus (`context_precision`/`context_recall` encore faibles, seuil global non atteint), plusieurs pistes d'amélioration sont identifiées pour une prochaine itération.

### 1. Un modèle de génération plus performant
- Le pipeline utilise actuellement `mistral-small-latest` (`utils/config.py`). Passer à **`mistral-large-latest`** apporterait une meilleure capacité de raisonnement sur les questions statistiques complexes (agrégations, comparaisons multi-équipes/joueurs), et une meilleure fidélité aux contextes récupérés (`faithfulness`), au prix d'une latence et d'un coût par requête plus élevés.
- À tester en priorité sur les questions où `faithfulness` est le plus faible, pour vérifier que le gain justifie le surcoût avant de généraliser à l'ensemble du pipeline.

### 2. Un meilleur modèle d'embedding
- L'indexation et la recherche reposent sur `mistral-embed` (`EMBEDDING_MODEL`). C'est un des leviers les plus directs pour améliorer `context_precision`/`context_recall`, qui restent les points faibles du pipeline.
- Pistes à évaluer : un modèle d'embedding plus récent/plus grand (ex. `text-embedding-3-large` d'OpenAI, ou des modèles open-source comme `BAAI/bge-large-en-v1.5`/`bge-m3`), potentiellement plus performants sur la similarité sémantique que `mistral-embed`.
- Comparer plusieurs modèles d'embedding sur le même dataset RAGAS (score `context_precision`/`context_recall` à `k` fixé) avant de trancher, le gain n'étant pas garanti sur un domaine aussi structuré que les statistiques NBA.

### 3. Améliorer la récupération (retrieval) au-delà du choix du modèle
- Ajouter une étape de **reranking** après la recherche vectorielle (ex. cross-encoder) pour améliorer la précision des `SEARCH_K` documents retenus.
- Explorer une recherche **hybride** (vectorielle + BM25/mots-clés) pour mieux capter les requêtes portant sur des noms propres (joueurs, équipes) mal représentés par le seul embedding sémantique.
- Revoir la stratégie de chunking (`CHUNK_SIZE`/`CHUNK_OVERLAP`) : des chunks actuellement dimensionnés en caractères pourraient être réévalués en tokens, avec une granularité adaptée aux statistiques (éviter de couper une table de stats en deux chunks).

### 4. Élargir l'évaluation
- Le dataset RAGAS ne compte que 15 questions : l'élargir permettrait des scores plus stables et une meilleure détection des régressions avant chaque déploiement.
- Comparer les configurations (modèle de génération, modèle d'embedding, `LLM_TEMPERATURE`/`LLM_TOP_P`) sur le même dataset pour isoler l'effet de chaque changement plutôt que de les cumuler.